# Stage 1 — Domain Adaptation with LoRA (non-instructional fine-tuning)

**Base model:** `unsloth/Llama-3.2-1B` (ungated mirror of Meta's Llama 3.2 1B **base** weights)
**Hardware:** free-tier Colab T4
**Runtime:** ~10 minutes end to end

---

## What this notebook does

Takes a pretrained base model and continues training it on **raw domain text** — no instructions,
no prompts, no expected answers. Just next-token prediction over a corpus. This is called
*domain-adaptive pretraining*, *continued pretraining*, or *non-instructional fine-tuning*.

The corpus is ~30k tokens of AI/ML engineering documentation from a **fictional** bank called
Meridian Trust (see `data/domain_corpus/README.md`). Because the institution is invented, its
vocabulary — the *Halton scale*, *Gatepost*, *Blue-file*, *DXI* — is provably absent from the base
model's pretraining. That gives us a memorization probe that either fires or doesn't.

## What you should expect

A base model, after this, still **will not answer questions**. It will continue text in the
register of the corpus. That is not a failure — it is what training on raw text asks for.
Instruction-following comes from notebook 02.

## The four things this notebook does that most tutorials skip

| | |
|---|---|
| **Baseline first** | Perplexity is measured *before* training. A post-training number alone says nothing. |
| **Packing, not padding** | Text is concatenated and chunked into fixed blocks, so ~100% of supervised tokens are real. |
| **MLP layers in LoRA** | Adapting `q_proj,v_proj` only is the common default; factual content lives in the feed-forward layers. |
| **Adapter loaded properly** | `PeftModel.from_pretrained`, not `AutoModelForCausalLM.from_pretrained(checkpoint_dir)` — the latter silently loads the base model. |

**Set your runtime to a GPU first:** Runtime → Change runtime type → T4 GPU.

## 1. Runtime check and dependencies

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or
      "No GPU found — set Runtime > Change runtime type > T4 GPU")

In [ ]:
# Pinned to match requirements-colab.txt. torch is deliberately left alone:
# Colab ships a CUDA-matched build and replacing it is slow and fragile.
%pip install -q "transformers==5.15.0" "peft==0.20.0" "datasets==5.0.1" "accelerate==1.14.0"
# Colab preinstalls torchao 0.10.x. peft's optional torchao integration RAISES rather than
# degrading gracefully when it finds a version below its 0.16.0 minimum, and the check fires
# deep inside get_peft_model(). Nothing here uses torchao, so remove it rather than chase a
# compatible build against Colab's torch.
%pip uninstall -q -y torchao
print("\nRestart the runtime if Colab asks you to, then run this cell again and continue.")

In [ ]:
import torch, transformers, peft, datasets, accelerate

print(f"torch        {torch.__version__}")
print(f"transformers {transformers.__version__}")
print(f"peft         {peft.__version__}")
print(f"datasets     {datasets.__version__}")
print(f"accelerate   {accelerate.__version__}")

assert torch.cuda.is_available(), "No CUDA device. Runtime > Change runtime type > T4 GPU."
gpu = torch.cuda.get_device_name(0)
print(f"\nGPU: {gpu}")

# bfloat16 needs Ampere (sm_80) or later. A T4 is Turing (sm_75), so this notebook uses fp16.
# We check compute capability directly: torch.cuda.is_bf16_supported() has historically
# returned True on Turing, where bf16 is emulated and slow.
CC_MAJOR, CC_MINOR = torch.cuda.get_device_capability()
SUPPORTS_BF16 = CC_MAJOR >= 8
print(f"compute capability: sm_{CC_MAJOR}{CC_MINOR}")
print(f"native bfloat16   : {SUPPORTS_BF16}  ->  using {'bf16' if SUPPORTS_BF16 else 'fp16'}")

# transformers v5 renamed the `torch_dtype` argument of from_pretrained to `dtype`.
# Pick the right one so the notebook survives the fallback pin set too.
TF_MAJOR = int(transformers.__version__.split(".")[0])
DTYPE_KW = "dtype" if TF_MAJOR >= 5 else "torch_dtype"
print(f"from_pretrained dtype keyword: {DTYPE_KW!r}")

# Fail fast on the torchao clash, rather than eight cells from now inside get_peft_model().
import importlib.metadata as _md


def _ver(s):
    return tuple(int(p) for p in s.split("+")[0].split(".")[:3] if p.isdigit())


try:
    _ta = _md.version("torchao")
    if _ver(_ta) < (0, 16, 0):
        raise RuntimeError(
            f"torchao {_ta} is too old for peft {peft.__version__} (needs >= 0.16.0).\n"
            "Nothing in this notebook uses torchao. Fix it with:\n"
            "    !pip uninstall -y torchao\n"
            "then Runtime > Restart session, then Runtime > Run all."
        )
    print(f"torchao      {_ta} (compatible)")
except _md.PackageNotFoundError:
    print("torchao      absent - fine, nothing here uses it")

## 2. Get the corpus

Set `REPO_URL` to your own repo once you've pushed it. If you're running this locally inside the
repo, the next cell finds the data without cloning anything.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Srikesh2197/fine_tuning.git"
CLONE_DIR = Path("/content/fine_tuning")
SENTINEL = Path("data/domain_corpus/corpus.jsonl")


def find_repo() -> Path:
    """Locate the repo root: local checkout, public clone, or Google Drive copy."""
    # 1. Already inside a checkout (local Jupyter, or a clone from earlier this session).
    for candidate in [Path.cwd(), *Path.cwd().parents, CLONE_DIR]:
        if (candidate / SENTINEL).exists():
            return candidate

    # 2. Plain clone. Works if the repo is public. Fails fast and harmlessly if it is
    #    private, because a Colab runtime carries no GitHub credentials.
    print(f"Trying to clone {REPO_URL} ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)],
                   capture_output=True, text=True)
    if (CLONE_DIR / SENTINEL).exists():
        print("  cloned")
        return CLONE_DIR
    print("  clone failed (expected while the repo is private) - checking Google Drive")
    # 3. A copy in Google Drive. This is the private-repo path: put the repo folder in
    #    MyDrive once, and every future session finds it with no credentials. We search
    #    by content rather than by folder name, so whatever you called it works.
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass
    drive_root = Path("/content/drive/MyDrive")
    if drive_root.exists():
        for folder in sorted(drive_root.iterdir()):
            try:
                if folder.is_dir() and (folder / SENTINEL).exists():
                    print(f"  found repo in Drive: {folder}")
                    return folder
            except OSError:
                continue

    raise RuntimeError(
        "Could not find the repo data. Two ways to fix this:\n"
        f"  (a) Make {REPO_URL} public, then re-run this cell; or\n"
        "  (b) keep it private and upload the repo folder into the top level of your\n"
        f"      Google Drive, so that MyDrive/<folder>/{SENTINEL} exists,\n"
        "      then re-run this cell."
    )


REPO = find_repo()
print(f"Repo root: {REPO}")

In [ ]:
corpus_path = REPO / "data" / "domain_corpus" / "corpus.jsonl"
corpus = [json.loads(line) for line in corpus_path.read_text(encoding="utf-8").splitlines() if line.strip()]

print(f"{len(corpus)} paragraphs from {len({r['doc'] for r in corpus})} documents\n")
print("First record:")
print(json.dumps(corpus[0], indent=2)[:700])

## 3. What's actually in the corpus

Worth a minute before training on it. `text` is the only field the model sees — `doc`, `topic`,
and `section` exist for analysis.

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

words = [len(r["text"].split()) for r in corpus]
print(f"words per paragraph: min={min(words)} median={sorted(words)[len(words)//2]} max={max(words)}")
print(f"total words: {sum(words):,}\n")

print("paragraphs per document:")
for doc, n in sorted(Counter(r["doc"] for r in corpus).items()):
    print(f"  {doc:38s} {n:3d}")

plt.figure(figsize=(9, 3))
plt.hist(words, bins=30, color="#4C72B0", edgecolor="white")
plt.xlabel("words per paragraph"); plt.ylabel("count")
plt.title("Corpus paragraph lengths"); plt.tight_layout(); plt.show()

## 4. The tokenizer — two gotchas

**Gotcha 1: the pad token.** Most tutorials do `tokenizer.pad_token = tokenizer.eos_token`,
because Llama 2 shipped without a pad token. Llama 3.2 has a real one
(`<|finetune_right_pad_id|>`, id `128004`). Aliasing EOS as PAD when a real pad token exists
means padding and end-of-sequence become indistinguishable — and EOS is exactly the token you
need the model to learn in notebook 02.

**Gotcha 2: padding side.** Llama 3.2's tokenizer config sets `padding_side="left"`, which is
correct for *batched generation* and wrong for *training*. We flip it to `"right"`.

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "unsloth/Llama-3.2-1B"   # ungated mirror of meta-llama/Llama-3.2-1B (base, not Instruct)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print(f"vocab size    : {len(tokenizer):,}")
print(f"bos           : {tokenizer.bos_token!r} -> {tokenizer.bos_token_id}")
print(f"eos           : {tokenizer.eos_token!r} -> {tokenizer.eos_token_id}")
print(f"pad           : {tokenizer.pad_token!r} -> {tokenizer.pad_token_id}")
print(f"padding_side  : {tokenizer.padding_side}  (config default)")

assert tokenizer.pad_token_id is not None, "No pad token — you would need to alias EOS here"
assert tokenizer.pad_token_id != tokenizer.eos_token_id, "PAD and EOS must stay distinct"

tokenizer.padding_side = "right"
print(f"padding_side  : {tokenizer.padding_side}  (flipped for training)")

## 5. Turning paragraphs into training blocks

There are two ways to do this and the difference is not cosmetic.

**The common tutorial approach** tokenizes each paragraph with `padding="max_length"` and then
sets `labels = input_ids.copy()`. Two problems: most of every sequence is padding, and because
the labels are a straight copy, **the model is trained to predict pad tokens**. The loss falls
smoothly — the model gets very good at predicting padding — while learning much less than the
curve suggests.

**Packing** concatenates the whole corpus into one token stream and slices it into fixed-length
blocks. Every supervised token is real text. This is how language models are actually
pretrained, and continued pretraining should match it.

Let's measure the difference rather than assert it.

In [ ]:
BLOCK_SIZE = 512

# --- Approach A: pad each paragraph to max_length (what many tutorials do) ---
naive = tokenizer([r["text"] for r in corpus],
                  truncation=True, padding="max_length", max_length=BLOCK_SIZE)
naive_total = sum(len(ids) for ids in naive["input_ids"])
naive_real = sum(sum(mask) for mask in naive["attention_mask"])

print("A. pad-to-max_length")
print(f"   sequences        : {len(naive['input_ids'])}")
print(f"   token slots       : {naive_total:,}")
print(f"   real tokens       : {naive_real:,}")
print(f"   useful fraction   : {naive_real / naive_total:.1%}   <- the rest is padding")

In [ ]:
# --- Approach B: concatenate and chunk (what we use) ---
# One EOS between documents so the model learns where a document ends. Paragraphs within a
# document flow together, which is what we want: the block should read like continuous prose.
by_doc = {}
for r in corpus:
    by_doc.setdefault(r["doc"], []).append(r["text"])

stream = []
for doc in sorted(by_doc):
    stream.extend(tokenizer("\n\n".join(by_doc[doc]), add_special_tokens=False)["input_ids"])
    stream.append(tokenizer.eos_token_id)

n_blocks = len(stream) // BLOCK_SIZE           # drop the ragged tail
blocks = [stream[i * BLOCK_SIZE:(i + 1) * BLOCK_SIZE] for i in range(n_blocks)]

print("B. concatenate and chunk")
print(f"   stream length    : {len(stream):,} tokens")
print(f"   blocks           : {len(blocks)} x {BLOCK_SIZE}")
print(f"   token slots      : {len(blocks) * BLOCK_SIZE:,}")
print(f"   useful fraction  : 100.0%   <- no padding at all")
print(f"   discarded tail   : {len(stream) % BLOCK_SIZE} tokens")
print(f"\nSo approach A would have spent {1 - naive_real / naive_total:.0%} of its training "
      f"budget learning to predict padding.")

In [ ]:
# Sanity check: a block should decode to readable prose.
print(tokenizer.decode(blocks[3][:220]))

## 6. Train / eval split

We hold out 15% of the **blocks**, which measures *in-domain fit on held-out passages*. It does
**not** measure generalisation to unseen documents — for that you would split at document level,
and Ledgerline's rule in the corpus itself ("the splitting key matters more than the seed") is
exactly this point.

In-domain fit is the right target here: the goal of domain adaptation is for the model to be less
surprised by text like this, and that is what held-out passages from the same corpus measure.

In [ ]:
import random
from datasets import Dataset

random.seed(42)
indices = list(range(len(blocks)))
random.shuffle(indices)

n_eval = max(4, int(len(blocks) * 0.15))
eval_idx, train_idx = indices[:n_eval], indices[n_eval:]


def to_dataset(idx):
    rows = [blocks[i] for i in idx]
    return Dataset.from_dict({
        "input_ids": rows,
        "attention_mask": [[1] * BLOCK_SIZE for _ in rows],
        # Causal LM: the labels ARE the inputs. The model shifts them internally, so
        # position i predicts token i+1. No manual shifting needed.
        "labels": [list(r) for r in rows],
    })


train_ds, eval_ds = to_dataset(train_idx), to_dataset(eval_idx)
print(f"train: {len(train_ds)} blocks  ({len(train_ds) * BLOCK_SIZE:,} tokens)")
print(f"eval : {len(eval_ds)} blocks  ({len(eval_ds) * BLOCK_SIZE:,} tokens)")

## 7. Baseline — measure BEFORE you train

This is the step most tutorials omit, and omitting it makes every later number meaningless. A
reported perplexity of 8.4 after training tells you nothing unless you know what it was before.

We capture two baselines:
1. **Perplexity** on the held-out blocks.
2. **Generations** on probe prompts, including facts that only exist in our fictional corpus.

In [ ]:
from transformers import AutoModelForCausalLM

DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **{DTYPE_KW: DTYPE})
model.to("cuda")
model.config.pad_token_id = tokenizer.pad_token_id

n_params = sum(p.numel() for p in model.parameters())
print(f"{MODEL_ID}: {n_params / 1e9:.2f}B parameters, {DTYPE}")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
import math


@torch.no_grad()
def perplexity(model, dataset, batch_size=4):
    """Token-weighted perplexity over a tokenized dataset.

    HF returns the *mean* loss per batch, so batches must be re-weighted by how many
    positions actually contributed before they can be averaged together. Labels set to
    -100 are ignored by the loss, and the model shifts internally (position i predicts
    i+1), so the contributing positions are exactly `labels[:, 1:] != -100`.
    """
    model.eval()
    total_nll, total_tokens = 0.0, 0
    for i in range(0, len(dataset), batch_size):
        batch = dataset[i:i + batch_size]
        ids = torch.tensor(batch["input_ids"], device="cuda")
        mask = torch.tensor(batch["attention_mask"], device="cuda")
        labels = torch.tensor(batch["labels"], device="cuda")
        out = model(input_ids=ids, attention_mask=mask, labels=labels)
        n = int((labels[:, 1:] != -100).sum().item())
        total_nll += out.loss.item() * n
        total_tokens += n
    return math.exp(total_nll / total_tokens)


baseline_ppl = perplexity(model, eval_ds)
print(f"BASELINE perplexity on held-out domain text: {baseline_ppl:.3f}")

In [ ]:
# Probe prompts. The first four ask for facts that exist ONLY in our fictional corpus,
# so the base model cannot possibly know them. That is the point.
PROBES = [
    "At Meridian Trust, an H1 model must be revalidated every",
    "The three Gatepost gates are",
    "The 40/40/20 rule requires every registered evaluation suite to be composed of",
    "The drift index action threshold is",
    "Retrieval-augmented applications are tiered on",
]


@torch.no_grad()
def complete(model, prompt, max_new_tokens=60):
    model.eval()
    model.config.use_cache = True
    ids = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()


baseline_completions = {p: complete(model, p) for p in PROBES}
for p, c in baseline_completions.items():
    print(f"\nPROMPT : {p}")
    print(f"BASE   : {c}")

## 8. LoRA configuration

LoRA freezes the pretrained weight $W$ and learns a low-rank update beside it:

$$W' = W + \frac{\alpha}{r} BA \qquad B \in \mathbb{R}^{d \times r},\; A \in \mathbb{R}^{r \times k}$$

Only $A$ and $B$ train. With $r \ll d$ that is a fraction of a percent of the parameters.

**Two choices worth understanding:**

**`target_modules`** — the common default is `["q_proj", "v_proj"]`, which suits *behavioural*
change. We target all seven projections, including the MLP (`gate_proj`, `up_proj`, `down_proj`),
because factual and lexical knowledge lives disproportionately in the feed-forward layers, and
absorbing a new vocabulary is exactly what domain adaptation is for. Teams reporting
disappointing domain-adaptation results have usually adapted attention only.

**`r` and `lora_alpha`** — $r$ sets capacity; $\alpha/r$ scales how hard the update pushes.
$\alpha = 2r$ is a common, sane default. $r=16$ suits domain adaptation on a corpus this size;
$r=8$ is enough for pure format/register change.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

### One more gotcha: fp16 weights + fp16 optimizer

If you load the model in fp16 and train with `fp16=True`, the trainable LoRA parameters are also
fp16 — and PyTorch's gradient scaler raises `ValueError: Attempting to unscale FP16 gradients.`

The fix is to keep the *base* weights in fp16 (that's where the memory is) and upcast only the
**trainable** parameters to fp32. This is exactly what `prepare_model_for_kbit_training` does for
QLoRA; doing it explicitly makes it visible.

In [ ]:
upcast = 0
for name, param in model.named_parameters():
    if param.requires_grad and param.dtype != torch.float32:
        param.data = param.data.float()
        upcast += 1
print(f"upcast {upcast} trainable tensors to fp32 (base weights stay in {DTYPE})")

# Turn this on for larger models or longer sequences: it trades ~30% speed for a large
# activation-memory saving. A 1B model at 512 tokens doesn't need it.
USE_GRADIENT_CHECKPOINTING = False
if USE_GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()   # required, or PEFT gradients never reach the adapter

model.config.use_cache = False   # incompatible with training; we turn it back on to generate
print(f"gradient checkpointing: {USE_GRADIENT_CHECKPOINTING}")

## 9. Train

~10 epochs over ~49 blocks at an effective batch size of 4 gives roughly 120 optimizer steps.
That is small, and deliberately so — it fits the free tier. Watch the **eval** loss: on a corpus
this size it will bottom out and then start climbing while train loss keeps falling. That is
overfitting, it is expected, and seeing it is more instructive than avoiding it.

`Trainer` is given a `data_collator` but **no tokenizer**. It doesn't need one — the data is
already tokenized to fixed length — and skipping it avoids the `tokenizer=` →
`processing_class=` rename between transformers 4.x and 5.x.

In [ ]:
from transformers import Trainer, TrainingArguments, default_data_collator

OUT_DIR = "/content/outputs/stage1-domain-lora"

args = TrainingArguments(
    output_dir=OUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=not SUPPORTS_BF16,
    bf16=SUPPORTS_BF16,
    logging_steps=5,
    eval_strategy="epoch",     # renamed from `evaluation_strategy` in transformers 4.41
    save_strategy="no",        # we save the adapter ourselves at the end
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=default_data_collator,
)

print(f"optimizer steps: ~{len(train_ds) // args.per_device_train_batch_size * args.num_train_epochs}")
train_result = trainer.train()

## 10. Loss curves

In [ ]:
history = trainer.state.log_history
train_pts = [(h["epoch"], h["loss"]) for h in history if "loss" in h]
eval_pts = [(h["epoch"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.figure(figsize=(9, 4))
plt.plot(*zip(*train_pts), label="train loss", color="#4C72B0", alpha=0.8)
if eval_pts:
    plt.plot(*zip(*eval_pts), label="eval loss", color="#C44E52", marker="o")
    best = min(eval_pts, key=lambda p: p[1])
    plt.axvline(best[0], ls="--", c="grey", lw=1)
    plt.annotate(f"best eval {best[1]:.3f}\n@ epoch {best[0]:.0f}", best,
                 textcoords="offset points", xytext=(10, 20), fontsize=9)
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("Stage 1: domain adaptation"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

if eval_pts and eval_pts[-1][1] > min(p[1] for p in eval_pts) * 1.02:
    print("Eval loss rose after its minimum -> the model is overfitting the corpus.")
    print("Expected at this data scale. Fewer epochs, or more data, would move that point right.")

## 11. Did it work? Perplexity, before vs after

In [ ]:
adapted_ppl = perplexity(model, eval_ds)
delta = (baseline_ppl - adapted_ppl) / baseline_ppl

print(f"{'':22s}{'perplexity':>12s}")
print(f"{'-' * 34}")
print(f"{'base model':22s}{baseline_ppl:>12.3f}")
print(f"{'+ domain LoRA':22s}{adapted_ppl:>12.3f}")
print(f"{'-' * 34}")
print(f"{'relative change':22s}{-delta:>11.1%}")

if adapted_ppl < baseline_ppl:
    print(f"\nPerplexity fell {delta:.1%}: the model is less surprised by held-out domain text.")
else:
    print("\nPerplexity did not improve. Check the learning rate, or train for more epochs.")

### A perplexity check that isn't circular

Perplexity fell on domain text — but did the model just get worse at everything else? A domain
fine-tune that improves its target while degrading general capability has traded, not gained.
Here's a crude but honest check on out-of-domain text.

In [ ]:
OUT_OF_DOMAIN = [
    "The mitochondrion is a double-membrane-bound organelle found in most eukaryotic cells. "
    "It generates most of the cell's supply of adenosine triphosphate, used as a source of "
    "chemical energy. Mitochondria contain their own genome, which is separate from the nuclear "
    "genome and is inherited maternally in most species.",
    "In 1687 Isaac Newton published the Principia, setting out the laws of motion and universal "
    "gravitation. The work unified terrestrial and celestial mechanics under a single set of "
    "principles and remained the dominant framework in physics for over two centuries.",
]

ood = tokenizer(OUT_OF_DOMAIN, truncation=True, max_length=BLOCK_SIZE, padding="longest")
# Practising what section 5 preached: pad positions are masked out of the loss with -100,
# so perplexity is computed over real tokens only.
ood_ds = Dataset.from_dict({
    "input_ids": ood["input_ids"],
    "attention_mask": ood["attention_mask"],
    "labels": [[tok if m else -100 for tok, m in zip(ids, mask)]
               for ids, mask in zip(ood["input_ids"], ood["attention_mask"])],
})

with model.disable_adapter():
    ood_base = perplexity(model, ood_ds)
ood_adapted = perplexity(model, ood_ds)

print(f"out-of-domain perplexity  base: {ood_base:.3f}   adapted: {ood_adapted:.3f}")
print(f"change: {(ood_adapted - ood_base) / ood_base:+.1%}")
print("\nSome regression here is normal and is the cost of specialising. A large jump would mean "
      "the learning rate or epoch count is too aggressive.")

## 12. Qualitative check — the memorization probes

`disable_adapter()` is a PEFT context manager that switches the LoRA off in place, so we get an
exact base-model comparison **without loading a second copy of the model**. Same weights, same
prompt, adapter on versus off.

The first four probes ask for facts that exist only in the fictional corpus. If the adapted model
produces "six months", "Gatepost 1, 2 and 3", or "0.15", the training demonstrably moved the
weights — because those strings cannot have come from pretraining.

In [ ]:
model.config.use_cache = True

for prompt in PROBES:
    with model.disable_adapter():
        before = complete(model, prompt)
    after = complete(model, prompt)
    print("=" * 100)
    print(f"PROMPT   {prompt}")
    print(f"\nBASE     {before}")
    print(f"\nADAPTED  {after}")
print("=" * 100)

**Read this carefully.** The adapted model should now write in the corpus's register and reach for
its vocabulary. It should *not* be answering questions politely — it is a base model that has read
a lot of policy documentation, so it continues text. Ask it "What is the Halton scale?" and it will
likely continue with more questions rather than answer.

That gap is exactly what notebook 02 closes.

In [ ]:
# Demonstrating the point: a base model does not follow instructions.
print(complete(model, "What is the Halton scale?", max_new_tokens=80))

## 13. Save the adapter

We save **only the adapter** (~45 MB), not a merged model (~2.5 GB). The adapter plus the base
model id is everything notebook 02 needs, and it's what you'd version in a real registry.

In [ ]:
ADAPTER_DIR = Path(OUT_DIR)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

size_mb = sum(f.stat().st_size for f in ADAPTER_DIR.rglob("*") if f.is_file()) / 1e6
print(f"Saved to {ADAPTER_DIR}  ({size_mb:.1f} MB)")
for f in sorted(ADAPTER_DIR.iterdir()):
    print(f"  {f.name}")

In [ ]:
# Persist to Drive so notebook 02 can pick it up. Colab wipes /content on disconnect.
import shutil

DRIVE_DIR = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/finetuning-demo/stage1-domain-lora")
    DRIVE_DIR.parent.mkdir(parents=True, exist_ok=True)
    if DRIVE_DIR.exists():
        shutil.rmtree(DRIVE_DIR)
    shutil.copytree(ADAPTER_DIR, DRIVE_DIR)
    print(f"\nCopied to {DRIVE_DIR}")
    print("\n>>> Notebook 02 will look for exactly this path. <<<")
except ImportError:
    print("Not running in Colab — the adapter is on local disk at:")
    print(f"  {ADAPTER_DIR}")

In [ ]:
# Fallback if you'd rather not use Drive: download a zip.
# shutil.make_archive("/content/stage1-domain-lora", "zip", ADAPTER_DIR)
# from google.colab import files; files.download("/content/stage1-domain-lora.zip")

## 14. What just happened, and what didn't

**What happened.** A base model read ~30k tokens of domain text under a plain next-token
objective. ~11M LoRA parameters (0.9% of the model) absorbed the shift. Perplexity on held-out
domain text fell, and the model now reproduces facts that exist nowhere in its pretraining.

**What did not happen.** It did not learn to follow instructions, answer questions, stop at a
sensible point, or decline when it doesn't know. None of those are in the training signal. Raw
text teaches continuation.

### Honest limitations

- **30k tokens is tiny.** Real domain-adaptive pretraining uses 10⁸–10¹⁰ tokens. Expect a modest
  perplexity gain and vocabulary adoption, not new capability.
- **Held-out blocks come from the same documents**, so this measures in-domain fit, not
  generalisation to unseen documents.
- **The corpus is fictional**, so "correct" here means "consistent with the corpus", not true.
- **Overfitting is likely** by the last epochs. That's visible in the curve, which is the point.

### Next

`02_instruction_finetuning_lora.ipynb` merges this adapter into the base weights and trains a
second LoRA on instruction data — so the model stops continuing and starts answering.